# Combined ClinVar + UniProt RAG — query & summarize

Pipeline: **ingest both sources** → **index both** → **combined search** → **summarize**.

Prerequisites:
- `python src/ingest_clinvar.py` then `python src/build_clinvar_index.py`
- `python src/ingest_uniprot.py` then `python src/build_uniprot_index.py`

Single-source notebooks: `clinvar_rag.ipynb`, `uniprot_rag.ipynb`.

In [12]:
# Natural-language query
query = "CFTR cystic fibrosis"

# "BRCA1 hereditary breast cancer pathogenic" → top 3 ClinVar variants (lowest distances)
# "CFTR cystic fibrosis" → mixed ranking: 2 ClinVar + 1 UniProt
# "mismatch repair mechanism" → top 3 UniProt variants 

In [13]:
# Optional: install dependencies if needed
# !pip install chromadb sentence-transformers transformers huggingface_hub accelerate

In [14]:
# Imports
from pathlib import Path
import sys

from sentence_transformers import SentenceTransformer

In [15]:
# Project paths and shared RAG helpers
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

from rag_search import search_combined
from rag_summary import load_llm, summarize

## Combined search

**Ranking:** Each source is queried independently (`TOP_K_CHUNKS` chunks per collection). Within each source, records are scored by their best (lowest-distance) chunk. Those per-record scores are merged and sorted globally; the top `TOP_K_RESULTS` records across ClinVar and UniProt are sent to the LLM (one full parquet row each).

In [16]:
# Retrieval limits
TOP_K_CHUNKS = 20  # chunk pool per collection; used for ranking (not sent to LLM)
TOP_K_RESULTS = 3  # top records globally across both sources

In [17]:
# Paths, embedding model, per-source search configs
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"

CLINVAR_PARQUET = data_processed / "clinvar_rag.parquet"
CLINVAR_COLLECTION = "clinvar_chunks"
CLINVAR_RECORD_FIELDS = [
    "VariationID", "Name", "GeneSymbol", "Chromosome", "Start", "Stop",
    "Type", "ClinicalSignificance", "ReviewStatus", "PhenotypeList",
    "PhenotypeIDS", "Assembly", "HGNC_ID", "NumberSubmitters", "LastEvaluated",
]

UNIPROT_PARQUET = data_processed / "uniprot_rag.parquet"
UNIPROT_COLLECTION = "protein_chunks"
UNIPROT_RECORD_FIELDS = [
    "Entry", "Entry Name", "Gene Names", "Protein names", "Length",
    "Function [CC]", "Involvement in disease", "Subcellular location [CC]",
    "Domain [FT]", "Region", "Zinc finger", "Domain [CC]",
    "Natural variant", "Polymorphism", "Tissue specificity",
    "Developmental stage", "Interacts with",
]

SEARCH_CONFIGS = {
    "clinvar": {
        "collection_name": CLINVAR_COLLECTION,
        "parquet_path": CLINVAR_PARQUET,
        "group_key": "variation_id",
        "record_id_column": "VariationID",
        "record_id_meta_key": "variation_id",
        "record_fields": CLINVAR_RECORD_FIELDS,
        "section_header_template": "### Result {rank} — variation_id={variation_id}, gene={gene}",
        "hit_fields": [("variation_id", "variation_id"), ("gene", "gene")],
        "full_record_label": "Full ClinVar record",
        "record_id_cast": int,
    },
    "uniprot": {
        "collection_name": UNIPROT_COLLECTION,
        "parquet_path": UNIPROT_PARQUET,
        "group_key": "entry_name",
        "record_id_column": "Entry",
        "record_id_meta_key": "accession",
        "record_fields": UNIPROT_RECORD_FIELDS,
        "section_header_template": "### Result {rank} — entry_name={entry_name}, accession={accession}, gene={gene}",
        "hit_fields": [("entry_name", "entry_name"), ("accession", "accession"), ("gene", "gene")],
        "full_record_label": "Full UniProt record",
    },
}

In [18]:
embed_model = SentenceTransformer(EMBED_MODEL)
result = search_combined(
    query,
    SEARCH_CONFIGS,
    top_k_chunks=TOP_K_CHUNKS,
    top_k_results=TOP_K_RESULTS,
    chroma_dir=chroma_dir,
    embed_model=embed_model,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection clinvar_chunks: 11591 chunks
Collection protein_chunks: 19201 chunks

COMBINED RETRIEVAL
Query: CFTR cystic fibrosis
  clinvar: top 20 chunks → 20 chunk(s) across 20 record(s)
  uniprot: top 20 chunks → 20 chunk(s) across 16 record(s)

Combined ranking: each record's best (lowest-distance) chunk is scored, then the top 3 records across all sources are sent to the LLM (one full parquet row per record).
Top 3 breakdown: clinvar=2, uniprot=1

TOP 3 RECORD(S) (combined cross-source ranking)
Preview below is the closest chunk only (truncated to 500 chars).

1. [clinvar] variation_id=7155 gene=CFTR best_chunk=summary dist=0.3983
Variation: 7155
Type: summary
gene: CFTR
pathogenicity: Pathogenic
impact: Cystic fibrosis. Cystic fibrosis;Hereditary pancreatitis;Bronchiectasis with or without elevated sweat chloride 1;Congenital bilateral aplasia of vas deferens from CFTR mutation. Cystic fibrosis;Congenital bilateral aplasia of vas deferens from CFTR mutation. Bronchiectasis with or 

## Summarize

In [19]:
SYSTEM_PROMPT = (
    "You are a clinical genomics and protein biology assistant. "
    "Write a brief summary in plain English sentences that answers the user's query using ONLY the ClinVar and UniProt records provided. "
    "Clearly distinguish variant evidence (ClinVar: cite VariationID and Name) from protein evidence (UniProt: cite Entry Name and Protein Name). "
    "Do not invent genes, variants, proteins, significance, or other facts not present in the provided text. "
    "If information is missing, say so briefly."
)

Load summarizer → generate answer from retrieved context only.

- **local:** downloads once → cached under `~/.cache/huggingface/`; 0.5B ~2GB RAM.
- **openai:** calls `gpt-4o-mini` via API; set `OPENAI_API_KEY` in `.env`.

The LLM receives the top 3 records from the combined ranking (ClinVar and/or UniProt), not the raw chunk pool.

In [20]:
# Summarization backend: "local" or "openai"
SUMMARIZER_BACKEND = "openai"
# Local model (SUMMARIZER_BACKEND == "local")
# Needs ~8GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# Needs ~4GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# OpenAI model (SUMMARIZER_BACKEND == "openai")
API_MODEL = "gpt-4o-mini"
MAX_NEW_TOKENS = 1024

In [21]:
if SUMMARIZER_BACKEND == "local":
    llm_model, llm_tokenizer = load_llm(
        LLM_MODEL,
        model=globals().get("llm_model"),
        tokenizer=globals().get("llm_tokenizer"),
    )
else:
    llm_model, llm_tokenizer = None, None

In [22]:
summary = summarize(
    query,
    result["context"],
    llm_model,
    llm_tokenizer,
    SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    backend=SUMMARIZER_BACKEND,
    api_model=API_MODEL,
)
print("=" * 60)
print(f"Query: {query}\n")
print(summary)

Summarized via OpenAI: gpt-4o-mini
Query: CFTR cystic fibrosis

Cystic fibrosis is associated with mutations in the CFTR gene, which encodes the cystic fibrosis transmembrane conductance regulator protein. Two specific pathogenic variants in the CFTR gene are noted: NM_000492.4(CFTR):c.1475C>T (p.Ser492Phe) (ClinVar VariationID: 7155) and NM_000492.4(CFTR):c.933C>G (p.Phe311Leu) (ClinVar VariationID: 7153). Both variants have been reviewed by expert panels and are linked to cystic fibrosis and related disorders.

The CFTR protein, known as the cystic fibrosis transmembrane conductance regulator (Entry Name: CFTR_HUMAN), functions as an epithelial ion channel that regulates ion and water transport across cell membranes. It plays a crucial role in maintaining fluid homeostasis in various tissues, including the respiratory system, where its dysfunction leads to the symptoms of cystic fibrosis.



1. query "BRCA1 hereditary breast cancer pathogenic"

Summarized via Qwen

The query "BRCA1 hereditary breast cancer pathogenic" indicates that there is a pathogenic variant in the BRCA1 gene on chromosome 17. The full ClinVar record for this variant shows that it is a single nucleotide polymorphism (SNP) with the change C -> T at position 241. This variant has been classified as pathogenic based on the review status and phenotype list provided.


Summarized via OpenAI: gpt-4o-mini

Query: BRCA1 hereditary breast cancer pathogenic

The BRCA1 gene has several pathogenic variants associated with hereditary breast cancer. Notable variants include:

1. **VariationID: 54565** - NM_007294.4(BRCA1):c.241C>T (p.Gln81Ter), which is classified as pathogenic and linked to hereditary breast ovarian cancer syndrome.
2. **VariationID: 55580** - NM_007294.4(BRCA1):c.5444G>A (p.Trp1815Ter), also classified as pathogenic and associated with familial breast and ovarian cancer.
3. **VariationID: 54957** - NM_007294.4(BRCA1):c.3661G>T (p.Glu1221Ter), which is similarly classified as pathogenic and related to hereditary breast ovarian cancer syndrome.

All these variants have been reviewed by expert panels and are significant in increasing the risk of breast and ovarian cancers.

2. Query: CFTR cystic fibrosis

Summarized via OpenAI: gpt-4o-mini

Cystic fibrosis is associated with mutations in the CFTR gene, which encodes the cystic fibrosis transmembrane conductance regulator protein. Two specific pathogenic variants in the CFTR gene are noted: NM_000492.4(CFTR):c.1475C>T (p.Ser492Phe) (ClinVar VariationID: 7155) and NM_000492.4(CFTR):c.933C>G (p.Phe311Leu) (ClinVar VariationID: 7153). Both variants have been reviewed by expert panels and are linked to cystic fibrosis and related disorders.

The CFTR protein, known as the cystic fibrosis transmembrane conductance regulator (Entry Name: CFTR_HUMAN), functions as an epithelial ion channel that regulates ion and water transport across cell membranes. It plays a crucial role in maintaining fluid homeostasis in various tissues, including the respiratory system, where its dysfunction leads to the symptoms of cystic fibrosis.